# M1a — TSP do Beerlink Distribuição (versão TEMPLATE)

**Treinamento de Otimização Aplicada — Genoa para Gradus**

> ⚠️ **Versão TEMPLATE.** Os dados (coordenadas, matriz de distâncias) já estão prontos. Você só precisa preencher o **modelo de otimização** nas seções marcadas com `# TODO`. Travou? A versão solução está em `m1_tsp_solution.ipynb`.

**Objetivo:** encontrar a ordem ótima de visita de 1 caminhão grande percorrendo o CD + 8 bares (capacidade ignorada nesta etapa).

**Modelo TSP com MTZ:**
- Variáveis: $x_{ij} \in \{0,1\}$ se o caminhão vai de $i$ a $j$; $u_i \ge 0$ para ordem da visita.
- FO: $\min \sum_{i,j} \text{dist}_{ij} x_{ij}$
- Restrições: saída 1 / nó, entrada 1 / nó, MTZ $u_i - u_j + n \cdot x_{ij} \le n-1$ para $i, j \ne 0,\ i \ne j$

## Setup (já pronto — apenas execute)

In [ ]:
%pip install -q ortools gurobipy

In [ ]:
import math, time
from itertools import product

# === DADOS DO CASO BEERLINK (já prontos — não mexer) =====================
COORDS = {
    'CD':            (-23.567, -46.685),
    'Centro':        (-23.553, -46.635),
    'Pinheiros':     (-23.565, -46.685),
    'Vila Madalena': (-23.555, -46.692),
    'Moema':         (-23.605, -46.665),
    'Tatuape':       (-23.539, -46.572),
    'Lapa':          (-23.521, -46.706),
    'Itaim':         (-23.583, -46.671),
    'Brooklin':      (-23.612, -46.690),
}
NODES = list(COORDS.keys())   # 9 nos: 0 = CD, 1..8 = bares
N = range(len(NODES))         # indices dos nos
n = len(NODES)

# Matriz de distancias 9x9 em km (Euclidiana lat-lon * 111)
def km(a, b):
    return math.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2) * 111

dist = [[km(COORDS[NODES[i]], COORDS[NODES[j]]) for j in N] for i in N]

print(f'{n} nos prontos. Distancia max: {max(dist[i][j] for i, j in product(N, N)):.1f} km')
print(f'Indices: N = range({n}), arcos validos = (i, j) com i != j')

## 1) Modele o TSP em OR-Tools

Preencha as 4 seções marcadas com `# TODO`. Roteiro:
1. Variáveis: dict `x[i, j]` binárias para arcos, lista `u[i]` contínuas para MTZ
2. Restrições: saída = 1, entrada = 1, MTZ
3. FO: minimizar $\sum \text{dist}_{ij} x_{ij}$
4. Resolver com `s.Solve()`

In [ ]:
from ortools.linear_solver import pywraplp

def solve_tsp_ortools():
    s = pywraplp.Solver.CreateSolver('CBC')

    # ── TODO 1 — Variáveis ──────────────────────────────────────
    # x = {(i, j): ... for i, j in product(N, N) if i != j}
    # u = [s.NumVar(0, n, f'u[{i}]') for i in N]
    x = ...   # dict de IntVar 0/1 indexado por (i, j) com i != j
    u = ...   # lista de NumVar entre 0 e n

    # ── TODO 2 — Restrições ─────────────────────────────────────
    # (a) cada nó tem 1 arco de saída: sum_j x[i,j] == 1 forall i
    # (b) cada nó tem 1 arco de entrada: sum_i x[i,j] == 1 forall j
    # (c) MTZ: u[i] - u[j] + n * x[i,j] <= n - 1  forall i,j != 0, i != j

    pass  # remova quando preencher

    # ── TODO 3 — FO ─────────────────────────────────────────────
    # s.Minimize(...)

    # ── TODO 4 — Resolver e extrair rota ────────────────────────
    # s.Solve()
    # Reconstrua a rota seguindo x[i, j].solution_value() > 0.5

    raise NotImplementedError('Preencha os 4 TODOs acima')

res_or = solve_tsp_ortools()
# Esperado: distancia total ~50 km, rota comecando e terminando em 'CD'

## 2) Modele o mesmo TSP em Gurobi

Mesmo modelo, sintaxe diferente. Use `m.addVars`, `m.addConstrs`, `gp.quicksum`.

In [ ]:
import gurobipy as gp
from gurobipy import GRB

def solve_tsp_gurobi():
    m = gp.Model('tsp')
    m.Params.OutputFlag = 0

    # ── TODO 1 — Variáveis (tupledict + indices) ────────────────
    # x = m.addVars([(i, j) for i, j in product(N, N) if i != j], vtype=GRB.BINARY, name='x')
    # u = m.addVars(N, lb=0, ub=n, name='u')
    x = ...
    u = ...

    # ── TODO 2 — Restrições (m.addConstrs com generator) ────────
    # (a) saída: m.addConstrs((x.sum(i, '*') == 1 for i in N), name='out')
    # (b) entrada: m.addConstrs((x.sum('*', j) == 1 for j in N), name='in')
    # (c) MTZ: m.addConstrs((u[i] - u[j] + n * x[i,j] <= n - 1 for i,j in product(N,N) if i != j and i != 0 and j != 0), name='mtz')

    # ── TODO 3 — FO ─────────────────────────────────────────────
    # m.setObjective(gp.quicksum(...), GRB.MINIMIZE)

    # ── TODO 4 — Resolver ───────────────────────────────────────
    # m.optimize()

    raise NotImplementedError('Preencha os 4 TODOs acima')

res_gb = solve_tsp_gurobi()

## 3) Compare os dois solvers

Os dois devem retornar **a mesma distância ótima** (TSP é problema unimodal). Os tempos podem variar.

In [ ]:
# TODO: monte um DataFrame ou print comparando res_or e res_gb
# Esperado: mesma distancia total (~50 km), tempos similares (<1 s ambos)
pass

## Pergunte ao Claude / Copilot se travar

Prompts úteis:
- *"Como construir a restrição MTZ em OR-Tools pywraplp?"*
- *"Por que MTZ precisa excluir o nó 0 (depósito)?"*
- *"Como reconstruir a sequência de visita a partir das variáveis $x_{ij}$?"*